In [1]:
# Cell 1: Setup, imports, and data load
#
# External validation against real PAN-OS firewall logs, re-aligned to follow
# the production scoring pipeline (dashboard/adaptrap_firewall_pipeline.py):
# a single frozen checkpoint (M5), three-tier ALLOW/ESCALATE_TO_ANALYST/BLOCK
# rules at the same percentiles, and the same quality metrics (flagging rate,
# anomaly score distribution, silhouette coefficient).
#
# - combined_logs_drop_type.csv : main dataset -> adapted + profiled + scored
#   by the model (simulates the "unlabeled" operational log the firewall sees).
# - combined_logs.csv           : same traffic, but includes the PAN "Type"/
#   "Action" columns -> used ONLY afterwards, as ground truth to score accuracy.

import sys
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, silhouette_score,
)

sys.path.insert(0, '/Users/gabriellecabangcala/thesis/AdapTrap/models')
from adaptrap_pipeline import (
    build_ip_profiles,
    get_feature_cols,
    compute_conn_rate,
    generate_rules,
)

# Same two-threshold config as the production pipeline
ESCALATE_PERCENTILE = 75
BLOCK_PERCENTILE = 95

MAIN_PATH = '/Users/gabriellecabangcala/thesis/AdapTrap/test/combined_logs_drop_type.csv'
TRUTH_PATH = '/Users/gabriellecabangcala/thesis/AdapTrap/test/combined_logs.csv'

df_main = pd.read_csv(MAIN_PATH, low_memory=False)
df_truth = pd.read_csv(TRUTH_PATH, low_memory=False)

print(f"Main (unlabeled) dataset: {df_main.shape}")
print(f"Truth (labeled) dataset:  {df_truth.shape}")


Main (unlabeled) dataset: (66579, 148)
Truth (labeled) dataset:  (66579, 149)


In [2]:
# Cell 2: Adapt the PAN-OS schema to the AdapTrap packet-level schema
#
# adaptrap_pipeline.build_ip_profiles()/compute_conn_rate() expect one row per
# packet with: source_ip, time_sec, protocol, length, dst_port, has_syn,
# has_ack, has_fin, has_rst, has_psh. PAN traffic logs are per-session, not
# per-packet, and their Flags column is a session bitmask (not TCP flag
# letters), so the TCP-flag ratios can't be recovered and are left at 0 -
# consistent with the adaptation used for the earlier PAN external-validation
# pass (test/run_validation.py).

def adapt_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={
        "Source address": "source_ip",
        "Destination Port": "dst_port",
        "Bytes": "length",
        "Application": "protocol",
    }).copy()

    df["time_sec"] = pd.to_datetime(df["Time Logged"], errors="coerce").astype("int64") // 10**9
    df = df.dropna(subset=["source_ip", "time_sec"])

    df["dst_port"] = pd.to_numeric(df["dst_port"], errors="coerce").fillna(0)
    df["length"] = pd.to_numeric(df["length"], errors="coerce").fillna(0)
    df["protocol"] = df["protocol"].fillna("other")

    for flag_col in ["has_syn", "has_ack", "has_fin", "has_rst", "has_psh"]:
        df[flag_col] = 0

    return df[["source_ip", "time_sec", "protocol", "length", "dst_port",
               "has_syn", "has_ack", "has_fin", "has_rst", "has_psh"]]

adapted_df = adapt_schema(df_main)
adapted_df = compute_conn_rate(adapted_df)
print(adapted_df.shape)
adapted_df.head()


(66579, 11)


,source_ip,time_sec,protocol,length,dst_port,has_syn,has_ack,has_fin,has_rst,has_psh,conn_rate
0,1.178.136.86,1790238,web-browsing,47316.0,443,0,0,0,0,0,1
1,1.187.66.21,1790214,web-browsing,27178.0,443,0,0,0,0,0,1
2,1.38.98.12,1790198,ssl,6285.0,443,0,0,0,0,0,1
3,1.4.192.61,1790152,web-browsing,3157.0,443,0,0,0,0,0,1
4,1.52.1.99,1790201,web-browsing,106947.0,443,0,0,0,0,0,1


In [3]:
# Cell 3: Build attacker IP profiles (same aggregation as training)

profiles = build_ip_profiles(adapted_df)
print(f"Behavioral IP profiles: {profiles.shape}")
profiles.head()


Behavioral IP profiles: (8969, 24)


,source_ip,total_packets,unique_dst_ports,protocol_diversity,avg_length,std_length,max_length,syn_ratio,ack_ratio,fin_ratio,...,session_duration,packets_per_second,pct_tcp,pct_vnc,pct_tlsv12,pct_sshv2,pct_dns,pct_tds,pct_telnet,pct_http
0,1.178.136.86,1.0,1.0,1.0,47316.0,0.0,47316.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.187.66.21,1.0,1.0,1.0,27178.0,0.0,27178.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.38.98.12,1.0,1.0,1.0,6285.0,0.0,6285.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.4.192.61,1.0,1.0,1.0,3157.0,0.0,3157.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.52.1.99,1.0,1.0,1.0,106947.0,0.0,106947.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Cell 4: Load the published M5 checkpoint (final/frozen, same as the
# production dashboard pipeline) and score every attacker IP profile from
# the external PAN dataset with the same three-tier rule engine.

MODEL_DIR = '/Users/gabriellecabangcala/thesis/AdapTrap/saved_models'
model = joblib.load(f'{MODEL_DIR}/m5_IF_model')
scaler = joblib.load(f'{MODEL_DIR}/m5_scaler')
FEATURE_COLUMNS = joblib.load(f'{MODEL_DIR}/m5_feature_cols')

# Align profile columns to the exact feature order the model was trained on;
# any pct_<protocol> column absent from this dataset's traffic mix is 0.
for col in FEATURE_COLUMNS:
    if col not in profiles.columns:
        profiles[col] = 0

X_scaled = scaler.transform(profiles[FEATURE_COLUMNS])

# No port lookup: enforcement blocks the full IP, not a specific port -
# matches the production pipeline exactly.
rules_df = generate_rules(
    model, X_scaled, profiles, {}, label="External Validation - PAN Firewall Logs (M5)",
    escalate_percentile=ESCALATE_PERCENTILE, block_percentile=BLOCK_PERCENTILE,
)



--- External Validation - PAN Firewall Logs (M5) ---
         source_ip  port  anomaly_score action
0  112.211.120.226  None       0.096108  BLOCK
1    103.151.28.82  None       0.092162  BLOCK
2    143.44.236.64  None       0.092090  BLOCK
3   138.84.137.210  None       0.092090  BLOCK
4   180.190.117.27  None       0.090367  BLOCK
5   112.207.208.91  None       0.089666  BLOCK
6    136.158.8.100  None       0.089571  BLOCK
7   112.200.33.214  None       0.089264  BLOCK
8   175.176.41.181  None       0.089146  BLOCK
9  112.210.227.115  None       0.088807  BLOCK

ALLOW: 4665, ESCALATE_TO_ANALYST: 3676, BLOCK: 628


/opt/homebrew/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.9.1. This might lead to bre

In [5]:
# Cell 5: Model-quality metrics, same definitions as the production pipeline
#
# silhouette_coefficient is computed from the model's own contamination-based
# decision (model.predict), matching Section 3.4.1 - NOT the percentile
# escalation threshold used by generate_rules. These are intentionally two
# different things: this reports model quality, generate_rules decides the
# operational firewall action.

def flagging_rate(actions: np.ndarray) -> float:
    return float(np.mean(actions != "ALLOW") * 100)


def anomaly_score_distribution(scores: np.ndarray) -> dict:
    return {
        "mean": float(np.mean(scores)),
        "std": float(np.std(scores)),
        "min": float(np.min(scores)),
        "max": float(np.max(scores)),
    }


def silhouette_coefficient(model, X: np.ndarray):
    flags = (model.predict(X) == -1).astype(int)
    if len(set(flags)) < 2:
        return None
    return float(silhouette_score(X, flags))


scores = rules_df["anomaly_score"].to_numpy()
actions = rules_df["action"].to_numpy()

metrics = {
    "flagging_rate_pct": round(flagging_rate(actions), 2),
    "anomaly_score_distribution": anomaly_score_distribution(scores),
    "silhouette_coefficient": silhouette_coefficient(model, X_scaled),
    "action_breakdown": {
        a: int((actions == a).sum()) for a in ["ALLOW", "ESCALATE_TO_ANALYST", "BLOCK"]
    },
}
metrics


{'flagging_rate_pct': 47.99,
 'anomaly_score_distribution': {'mean': -0.0035358656632269465,
  'std': 0.03411344078804944,
  'min': -0.11835751050490045,
  'max': 0.09610839031138052},
 'silhouette_coefficient': 0.09182430785892928,
 'action_breakdown': {'ALLOW': 4665,
  'ESCALATE_TO_ANALYST': 3676,
  'BLOCK': 628}}

In [6]:
# Cell 6: Side-by-side model action vs PAN ground truth
#
# PAN ground truth is aggregated per source IP. Any reset-both, alert, or
# THREAT record is treated as BLOCK; all other traffic is treated as ALLOW.

truth_rows = df_truth[["Source address", "Type", "Action"]].copy()
truth_rows["Source address"] = truth_rows["Source address"].astype(str)
truth_rows["Action"] = truth_rows["Action"].fillna("").str.lower()
truth_rows["Type"] = truth_rows["Type"].fillna("").str.upper()

truth_rows["ground_truth_action"] = np.where(
    truth_rows["Action"].isin(["reset-both", "alert"])
    | truth_rows["Type"].eq("THREAT"),
    "BLOCK",
    "ALLOW",
)

action_priority = {"ALLOW": 0, "BLOCK": 1}
truth_rows["action_priority"] = truth_rows["ground_truth_action"].map(action_priority)
truth_by_ip = (
    truth_rows.sort_values("action_priority")
    .drop_duplicates("Source address", keep="last")
    [["Source address", "ground_truth_action"]]
    .rename(columns={"Source address": "source_ip"})
)

comparison_df = rules_df[["source_ip", "anomaly_score", "action"]].merge(
    truth_by_ip, on="source_ip", how="left"
)
comparison_df["ground_truth_action"] = comparison_df["ground_truth_action"].fillna("ALLOW")
comparison_df["match"] = (
    comparison_df["action"] == comparison_df["ground_truth_action"]
)
comparison_df = comparison_df.rename(columns={
    "source_ip": "Source IP",
    "anomaly_score": "Anomaly Score",
    "action": "Model Action",
    "ground_truth_action": "Ground Truth Action",
    "match": "Match",
})

print("Side-by-side comparison:")
display(comparison_df.sort_values("Anomaly Score", ascending=False).head(30))

comparison_df.to_csv(
    "/Users/gabriellecabangcala/thesis/AdapTrap/test/validation_comparison.csv",
    index=False,
)

comparison_labels = ["ALLOW", "BLOCK"]
print(f"\nIPs evaluated: {len(comparison_df)}")
print(f"Exact action agreement: {comparison_df['Match'].mean():.2%}")
print("\nConfusion Matrix [rows=truth, cols=pred] (ALLOW, BLOCK):")
print(confusion_matrix(
    comparison_df["Ground Truth Action"],
    comparison_df["Model Action"],
    labels=comparison_labels,
))
print("\nClassification Report:")
print(classification_report(
    comparison_df["Ground Truth Action"],
    comparison_df["Model Action"],
    labels=comparison_labels,
    zero_division=0,
))

Side-by-side comparison:


,Source IP,Anomaly Score,Model Action,Ground Truth Action,Match
0,112.211.120.226,0.096108,BLOCK,ALLOW,False
1,103.151.28.82,0.092162,BLOCK,ALLOW,False
2,143.44.236.64,0.092090,BLOCK,ALLOW,False
3,138.84.137.210,0.092090,BLOCK,ALLOW,False
4,180.190.117.27,0.090367,BLOCK,ALLOW,False
5,112.207.208.91,0.089666,BLOCK,ALLOW,False
6,136.158.8.100,0.089571,BLOCK,ALLOW,False
7,112.200.33.214,0.089264,BLOCK,ALLOW,False
8,175.176.41.181,0.089146,BLOCK,ALLOW,False
9,112.210.227.115,0.088807,BLOCK,ALLOW,False



IPs evaluated: 8969
Exact action agreement: 51.66%

Confusion Matrix [rows=truth, cols=pred] (ALLOW, BLOCK):
[[4632  627]
 [  33    1]]

Classification Report:
              precision    recall  f1-score   support

       ALLOW       0.99      0.52      0.68      8934
       BLOCK       0.00      0.03      0.00        35

   micro avg       0.88      0.52      0.65      8969
   macro avg       0.50      0.27      0.34      8969
weighted avg       0.99      0.52      0.68      8969



In [7]:
# Cell 7: Optional 70/15/15 split of the external profiles
#
# This split is for a separate holdout report only. The saved M5 model is not
# retrained, refit, or retuned on any part of the Palo Alto dataset.

from sklearn.model_selection import train_test_split

external_indices = np.arange(len(profiles))
external_train_idx, external_temp_idx = train_test_split(
    external_indices, test_size=0.30, random_state=42
)
external_val_idx, external_holdout_idx = train_test_split(
    external_temp_idx, test_size=0.50, random_state=42
)

print(f"External split: {len(external_train_idx)} train / "
      f"{len(external_val_idx)} validation / "
      f"{len(external_holdout_idx)} holdout")


External split: 6278 train / 1345 validation / 1346 holdout


In [8]:
# Cell 8: Side-by-side comparison with holdout summary metrics

holdout_profiles = profiles.iloc[external_holdout_idx].copy()
holdout_X = scaler.transform(holdout_profiles[FEATURE_COLUMNS])
holdout_rules = generate_rules(
    model, holdout_X, holdout_profiles, {},
    label="",
    escalate_percentile=ESCALATE_PERCENTILE,
    block_percentile=BLOCK_PERCENTILE,
)

holdout_truth = df_truth[["Source address", "Type", "Action"]].copy()
holdout_truth["Source address"] = holdout_truth["Source address"].astype(str)
holdout_truth["Action"] = holdout_truth["Action"].fillna("").str.lower()
holdout_truth["Type"] = holdout_truth["Type"].fillna("").str.upper()
holdout_truth["ground_truth_action"] = np.where(
    holdout_truth["Action"].isin(["reset-both", "alert"])
    | holdout_truth["Type"].eq("THREAT"),
    "BLOCK",
    "ALLOW",
)
holdout_truth["action_priority"] = holdout_truth["ground_truth_action"].map(
    {"ALLOW": 0, "BLOCK": 1}
)
truth_by_ip_holdout = (
    holdout_truth.sort_values("action_priority")
    .drop_duplicates("Source address", keep="last")
    [["Source address", "ground_truth_action"]]
    .rename(columns={"Source address": "source_ip"})
)

holdout_comparison = holdout_rules[["source_ip", "anomaly_score", "action"]].merge(
    truth_by_ip_holdout, on="source_ip", how="left"
)
holdout_comparison["ground_truth_action"] = holdout_comparison["ground_truth_action"].fillna("ALLOW")
holdout_comparison["match"] = (
    holdout_comparison["action"] == holdout_comparison["ground_truth_action"]
)
holdout_comparison = holdout_comparison.rename(columns={
    "source_ip": "Source IP",
    "anomaly_score": "Anomaly Score",
    "action": "Model Action",
    "ground_truth_action": "Ground Truth Action",
    "match": "Match",
})
holdout_comparison.to_csv(
    "/Users/gabriellecabangcala/thesis/AdapTrap/test/holdout_validation_comparison.csv",
    index=False,
)

print("Side-by-side holdout comparison:")
display(holdout_comparison.sort_values("Anomaly Score", ascending=False).head(30))

correct_matches = int(holdout_comparison["Match"].sum())
accuracy = holdout_comparison["Match"].mean()

print(f"Total IPs evaluated: {len(holdout_comparison)}")
print(f"Correct matches: {correct_matches}")
print(f"Accuracy: {accuracy:.2%}")

print("Confusion Matrix [rows=truth, cols=pred] (ALLOW, ESCALATE, BLOCK):")
print(pd.crosstab(
    pd.Categorical(holdout_comparison["Ground Truth Action"], categories=["ALLOW", "BLOCK"]),
    pd.Categorical(
        holdout_comparison["Model Action"],
        categories=["ALLOW", "ESCALATE_TO_ANALYST", "BLOCK"],
    ),
))

Side-by-side holdout comparison:


,Source IP,Anomaly Score,Model Action,Ground Truth Action,Match
0,103.151.28.82,0.092162,BLOCK,ALLOW,False
1,180.190.117.27,0.090367,BLOCK,ALLOW,False
2,112.207.208.91,0.089666,BLOCK,ALLOW,False
4,209.35.161.173,0.087746,BLOCK,ALLOW,False
3,112.204.175.76,0.087746,BLOCK,ALLOW,False
5,88.103.27.122,0.087291,BLOCK,ALLOW,False
6,124.6.184.152,0.087291,BLOCK,ALLOW,False
7,34.34.253.166,0.087291,BLOCK,ALLOW,False
8,180.190.173.147,0.087291,BLOCK,ALLOW,False
9,112.201.129.104,0.087291,BLOCK,ALLOW,False


Total IPs evaluated: 1346
Correct matches: 710
Accuracy: 52.75%
Confusion Matrix [rows=truth, cols=pred] (ALLOW, ESCALATE, BLOCK):
col_0  ALLOW  ESCALATE_TO_ANALYST  BLOCK
row_0                                   
ALLOW    710                  528     99
BLOCK      9                    0      0


In [9]:
# Cell 9: Evaluate the frozen M1 checkpoint on the same 70/15/15 external split
# The split is used only for reporting; M1 is not trained or tuned on PAN data.

m1_model = joblib.load(f'{MODEL_DIR}/m1_IF_model')
m1_scaler = joblib.load(f'{MODEL_DIR}/m1_scaler')
m1_feature_columns = joblib.load(f'{MODEL_DIR}/m1_feature_cols')

m1_profiles = profiles.copy()
for col in m1_feature_columns:
    if col not in m1_profiles.columns:
        m1_profiles[col] = 0

m1_splits = {
    "train": external_train_idx,
    "validation": external_val_idx,
    "holdout": external_holdout_idx,
}
m1_split_results = []
m1_holdout_comparison = None

for split_name, split_indices in m1_splits.items():
    split_profiles = m1_profiles.iloc[split_indices].copy()
    split_X = m1_scaler.transform(split_profiles[m1_feature_columns])
    split_rules = generate_rules(
        m1_model, split_X, split_profiles, {},
        label=f"M1 External Validation - {split_name}",
        escalate_percentile=ESCALATE_PERCENTILE,
        block_percentile=BLOCK_PERCENTILE,
    )
    split_comparison = split_rules[["source_ip", "anomaly_score", "action"]].merge(
        truth_by_ip, on="source_ip", how="left"
    )
    split_comparison["ground_truth_action"] = split_comparison[
        "ground_truth_action"
    ].fillna("ALLOW")
    split_comparison["match"] = (
        split_comparison["action"] == split_comparison["ground_truth_action"]
    )

    m1_split_results.append({
        "split": split_name,
        "profiles": len(split_comparison),
        "accuracy_pct": round(accuracy_score(
            split_comparison["ground_truth_action"], split_comparison["action"]
        ) * 100, 2),
        "allow": int((split_comparison["action"] == "ALLOW").sum()),
        "escalate": int((split_comparison["action"] == "ESCALATE_TO_ANALYST").sum()),
        "block": int((split_comparison["action"] == "BLOCK").sum()),
    })

    if split_name == "holdout":
        m1_holdout_comparison = split_comparison.rename(columns={
            "source_ip": "Source IP",
            "anomaly_score": "Anomaly Score",
            "action": "Model Action",
            "ground_truth_action": "Ground Truth Action",
            "match": "Match",
        })
        m1_holdout_comparison.to_csv(
            "/Users/gabriellecabangcala/thesis/AdapTrap/test/m1_holdout_validation_comparison.csv",
            index=False,
        )

m1_split_summary = pd.DataFrame(m1_split_results)
m1_holdout_accuracy_score = accuracy_score(
    m1_holdout_comparison["Ground Truth Action"],
    m1_holdout_comparison["Model Action"],
)
m1_correct_matches = int(m1_holdout_comparison["Match"].sum())
m1_confusion = pd.crosstab(
    pd.Categorical(
        m1_holdout_comparison["Ground Truth Action"],
        categories=["ALLOW", "BLOCK"],
    ),
    pd.Categorical(
        m1_holdout_comparison["Model Action"],
        categories=["ALLOW", "ESCALATE_TO_ANALYST", "BLOCK"],
    ),
)
m1_confusion.index.name = "Ground Truth"
m1_confusion.columns.name = "Model Action"

print("M1 70/15/15 external evaluation (frozen checkpoint):")
display(m1_split_summary)
print(f"\nM1 holdout profiles: {len(m1_holdout_comparison)}")
print(f"Correct exact action matches: {m1_correct_matches}")
print(
    f"Exact action-agreement accuracy: {m1_holdout_accuracy_score:.4f} "
    f"({m1_holdout_accuracy_score:.2%})"
)
print("\nConfusion matrix (rows=ground truth, columns=model action):")
display(m1_confusion)
print("\nM1 holdout comparison, highest anomaly scores:")
display(m1_holdout_comparison.sort_values("Anomaly Score", ascending=False).head(30))


--- M1 External Validation - train ---
         source_ip  port  anomaly_score action
0  112.211.120.226  None       0.130795  BLOCK
1   175.176.41.181  None       0.127320  BLOCK
2   138.84.137.210  None       0.121673  BLOCK
3   112.204.171.52  None       0.120703  BLOCK
4   209.35.164.212  None       0.116103  BLOCK
5   126.52.131.246  None       0.113211  BLOCK
6   112.211.112.21  None       0.112986  BLOCK
7      27.49.15.67  None       0.112986  BLOCK
8   112.211.112.44  None       0.112986  BLOCK
9     158.62.16.14  None       0.112529  BLOCK

ALLOW: 3195, ESCALATE_TO_ANALYST: 2687, BLOCK: 396

--- M1 External Validation - validation ---
         source_ip  port  anomaly_score action
0    143.44.236.64  None       0.125012  BLOCK
1   131.226.101.24  None       0.115271  BLOCK
2   110.54.132.233  None       0.112986  BLOCK
3     13.52.79.116  None       0.112846  BLOCK
4  124.107.130.194  None       0.112529  BLOCK
5     136.158.8.33  None       0.111618  BLOCK
6   42.115.137.22

,split,profiles,accuracy_pct,allow,escalate,block
0,train,6278,50.59,3195,2687,396
1,validation,1345,51.00,689,571,85
2,holdout,1346,51.78,706,554,86



M1 holdout profiles: 1346
Correct exact action matches: 697
Exact action-agreement accuracy: 0.5178 (51.78%)

Confusion matrix (rows=ground truth, columns=model action):


Model Action,ALLOW,ESCALATE_TO_ANALYST,BLOCK
Ground Truth,,,
ALLOW,697,554,86
BLOCK,9,0,0



M1 holdout comparison, highest anomaly scores:


,Source IP,Anomaly Score,Model Action,Ground Truth Action,Match
0,180.190.117.27,0.130326,BLOCK,ALLOW,False
1,112.211.113.55,0.127593,BLOCK,ALLOW,False
2,202.57.49.228,0.117234,BLOCK,ALLOW,False
3,112.207.208.91,0.116188,BLOCK,ALLOW,False
4,138.84.138.168,0.115271,BLOCK,ALLOW,False
5,180.190.166.183,0.115271,BLOCK,ALLOW,False
6,88.103.27.122,0.112529,BLOCK,ALLOW,False
7,110.235.131.235,0.112529,BLOCK,ALLOW,False
9,112.206.71.70,0.111618,BLOCK,ALLOW,False
10,180.190.173.147,0.111618,BLOCK,ALLOW,False


In [10]:
# Cell 10: Compare exact action-agreement accuracy across the three evaluations
# The full pass evaluates all external profiles; the other two use the same holdout split.

accuracy_comparison = pd.DataFrame([
    {
        "Evaluation": "M5 - full external pass",
        "Correct exact matches": int(comparison_df["Match"].sum()),
        "Profiles evaluated": len(comparison_df),
        "Accuracy (%)": comparison_df["Match"].mean() * 100,
    },
    {
        "Evaluation": "M5 - 70/15/15 holdout",
        "Correct exact matches": int(holdout_comparison["Match"].sum()),
        "Profiles evaluated": len(holdout_comparison),
        "Accuracy (%)": holdout_comparison["Match"].mean() * 100,
    },
    {
        "Evaluation": "M1 - 70/15/15 holdout",
        "Correct exact matches": int(m1_holdout_comparison["Match"].sum()),
        "Profiles evaluated": len(m1_holdout_comparison),
        "Accuracy (%)": m1_holdout_comparison["Match"].mean() * 100,
    },
])

accuracy_comparison["Accuracy (%)"] = accuracy_comparison["Accuracy (%)"].round(2)
print("Accuracy comparison:")
display(accuracy_comparison)

m5_vs_m1_pp = (
    holdout_comparison["Match"].mean()
    - m1_holdout_comparison["Match"].mean()
) * 100


Accuracy comparison:


,Evaluation,Correct exact matches,Profiles evaluated,Accuracy (%)
0,M5 - full external pass,4633,8969,51.66
1,M5 - 70/15/15 holdout,710,1346,52.75
2,M1 - 70/15/15 holdout,697,1346,51.78
